# Upstox API Quick Reference

This notebook provides quick access to all standard Upstox API functions for testing and exploring response formats.

**Official Documentation:** https://github.com/upstox/upstox-python

## Setup & Authentication

In [23]:
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

# Import all helper libraries
from lib.core.authentication import check_existing_token, perform_authentication, save_access_token
from lib.api.market_data import download_nse_market_data, get_market_quotes, get_vwap
from lib.api.historical import get_historical_data, get_intraday_data_v3
from lib.api.option_chain import (
    get_expiries, get_nearest_expiry, get_option_chain_dataframe,
    get_greeks, get_market_data as get_option_market_data,
    get_atm_strike_from_chain, get_atm_iv
)
from lib.api.market_quotes import (
    get_ltp_quote, get_multiple_ltp_quotes,
    get_ohlc_quote, get_option_greek, get_multiple_option_greeks
)
from lib.utils.instrument_utils import get_future_instrument_key, get_option_instrument_key

import pandas as pd
from datetime import datetime

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [24]:
# Load Access Token
token_path = os.path.join('..', 'lib', 'core', 'accessToken.txt')

if os.path.exists(token_path):
    with open(token_path, 'r') as f:
        access_token = f.read().strip()
    print(f"✅ Token loaded: {access_token[:20]}...")
else:
    print("❌ Token not found. Run authentication first.")
    # Uncomment to authenticate:
    # access_token = perform_authentication()
    # save_access_token(access_token)

✅ Token loaded: eyJ0eXAiOiJKV1QiLCJr...


## 1. Market Data - NSE Instruments

In [25]:
# Download NSE Market Data (Instrument Master)
nse_data = download_nse_market_data()

print(f"Total Instruments: {len(nse_data)}")
print(f"\nColumns: {list(nse_data.columns)}")
print(f"\nSample Data:")
nse_data.head()

NSE market data loaded successfully. Shape: (100235, 26)
Total Instruments: 100235

Columns: ['weekly', 'segment', 'name', 'exchange', 'expiry', 'instrument_type', 'asset_symbol', 'underlying_symbol', 'instrument_key', 'lot_size', 'freeze_quantity', 'exchange_token', 'minimum_lot', 'tick_size', 'asset_type', 'underlying_type', 'trading_symbol', 'strike_price', 'qty_multiplier', 'isin', 'security_type', 'asset_key', 'underlying_key', 'short_name', 'last_trading_date', 'price_quote_unit']

Sample Data:


,weekly,segment,name,exchange,expiry,instrument_type,asset_symbol,underlying_symbol,instrument_key,lot_size,...,trading_symbol,strike_price,qty_multiplier,isin,security_type,asset_key,underlying_key,short_name,last_trading_date,price_quote_unit
0,0.0,NCD_FO,JPYINR,NSE,1.774463e+12,CE,JPYINR,JPYINR,NCD_FO|14294,1.0,...,JPYINR 61 CE 25 MAR 26,61.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,NCD_FO,JPYINR,NSE,1.774463e+12,PE,JPYINR,JPYINR,NCD_FO|14295,1.0,...,JPYINR 61 PE 25 MAR 26,61.0,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NSE_EQ,SDL RJ 7.49% 2035,NSE,NaN,SG,NaN,NaN,NSE_EQ|IN2920250163,100.0,...,749RJ35,NaN,1.0,IN2920250163,NORMAL,NaN,NaN,NaN,NaN,NaN
3,NaN,NSE_EQ,GOI TBILL 182D-14/05/26,NSE,NaN,TB,NaN,NaN,NSE_EQ|IN002025Y339,100.0,...,182D140526,NaN,1.0,IN002025Y339,NORMAL,NaN,NaN,NaN,NaN,NaN
4,NaN,NSE_EQ,GOI TBILL 182D-08/05/26,NSE,NaN,TB,NaN,NaN,NSE_EQ|IN002025Y321,100.0,...,182D080526,NaN,1.0,IN002025Y321,NORMAL,NaN,NaN,NaN,NaN,NaN


## 2. Futures Data

In [26]:
# Get Nifty Futures Instrument Key
nifty_fut_key = get_future_instrument_key("NIFTY", nse_data)
print(f"Nifty Future Key: {nifty_fut_key}")

Nifty Future Key: NSE_FO|49229


In [32]:
# Get VWAP (Average Traded Price) for Futures
vwap = get_vwap(access_token, nifty_fut_key)
print(f"Nifty Futures VWAP: ₹{vwap:.2f}")

Nifty Futures VWAP: ₹25201.08


Tick: {'type': 'market_info', 'currentTs': '1769178600057', 'marketInfo': {'segmentStatus': {'US_EQ': 'NORMAL_OPEN'}}}


In [28]:
# Get Full Quote for Futures
from lib.api.market_data import get_market_quote_for_instrument

quote = get_market_quote_for_instrument(access_token, nifty_fut_key)
if quote:
    print(f"LTP: ₹{quote.get('last_price', 0):.2f}")
    print(f"VWAP: ₹{quote.get('average_price', 0):.2f}")
    print(f"Volume: {quote.get('volume', 0):,}")
    print(f"OI: {quote.get('oi', 0):,}")
    print(f"\nOHLC: {quote.get('ohlc', {})}")

LTP: ₹25080.00
VWAP: ₹25201.08
Volume: 7,055,490
OI: 12,757,355.0

OHLC: {'open': 25379.9, 'high': 25379.9, 'low': 25045.7, 'close': 25080.0}


## 3. Historical & Intraday Data

In [29]:
# Get Intraday 1-minute candles
candles_1min = get_intraday_data_v3(access_token, nifty_fut_key, "minute", 1)

if candles_1min:
    df_1min = pd.DataFrame(candles_1min)
    print(f"Total 1-min candles: {len(df_1min)}")
    print(f"\nColumns: {list(df_1min.columns)}")
    print(f"\nLatest 5 candles:")
    display(df_1min.tail())

Total 1-min candles: 375

Columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume']

Latest 5 candles:


,timestamp,open,high,low,close,volume
370,2026-01-23T15:25:00+05:30,25085.0,25088.7,25082.0,25085.0,16965
371,2026-01-23T15:26:00+05:30,25085.0,25086.0,25075.0,25081.8,32370
372,2026-01-23T15:27:00+05:30,25080.6,25086.0,25079.7,25085.0,20670
373,2026-01-23T15:28:00+05:30,25085.7,25086.8,25079.9,25080.5,28080
374,2026-01-23T15:29:00+05:30,25080.5,25089.0,25076.7,25080.0,38350


In [30]:
# Get Intraday 5-minute candles
candles_5min = get_intraday_data_v3(access_token, nifty_fut_key, "minute", 5)

if candles_5min:
    df_5min = pd.DataFrame(candles_5min)
    print(f"Total 5-min candles: {len(df_5min)}")
    display(df_5min.tail())

Total 5-min candles: 75


,timestamp,open,high,low,close,volume
70,2026-01-23T15:05:00+05:30,25097.9,25100.0,25045.7,25060.0,141895
71,2026-01-23T15:10:00+05:30,25060.6,25084.0,25056.0,25075.0,122850
72,2026-01-23T15:15:00+05:30,25077.9,25105.0,25060.0,25076.4,146120
73,2026-01-23T15:20:00+05:30,25078.8,25085.0,25072.1,25084.4,118625
74,2026-01-23T15:25:00+05:30,25085.0,25089.0,25075.0,25080.0,136435


In [31]:
# Get Historical Daily Data (Last 30 days)
hist_data = get_historical_data(access_token, nifty_fut_key, "day", 30)

if hist_data:
    df_hist = pd.DataFrame(hist_data)
    print(f"Total daily candles: {len(df_hist)}")
    display(df_hist.tail())

❌ V3 API Error (400): {"status":"error","errors":[{"errorCode":"UDAPI1146","message":"Invalid unit","propertyPath":"getHistoricalCandleData.unit","invalidValue":"day","error_code":"UDAPI1146","property_path":"getHistoricalCandleData.unit","invalid_value":"day"}]}


## 4. Option Chain Data

In [11]:
# Get Available Expiries
expiries = get_expiries(access_token, "NSE_INDEX|Nifty 50")
print(f"Available Expiries: {expiries}")

# Get Nearest Expiry
nearest_expiry = get_nearest_expiry(access_token, "NSE_INDEX|Nifty 50")
print(f"\nNearest Expiry: {nearest_expiry}")

Available Expiries: ['2026-01-27', '2026-02-03', '2026-02-10', '2026-02-17', '2026-02-24', '2026-03-02', '2026-03-30', '2026-06-30', '2026-09-29', '2026-12-29', '2027-06-29', '2027-12-28', '2028-06-27', '2028-12-26', '2029-06-26', '2029-12-24', '2030-06-25', '2030-12-31']

Nearest Expiry: 2026-01-27


In [12]:
# Get Full Option Chain as DataFrame
chain_df = get_option_chain_dataframe(access_token, "NSE_INDEX|Nifty 50", nearest_expiry)

if chain_df is not None and not chain_df.empty:
    print(f"Total Strikes: {len(chain_df)}")
    print(f"Spot Price: ₹{chain_df['spot_price'].iloc[0]:.2f}")
    print(f"PCR: {chain_df['pcr'].iloc[0]:.2f}")
    print(f"\nColumns: {list(chain_df.columns)}")
    print(f"\nSample Data (ATM ± 5 strikes):")
    
    atm = get_atm_strike_from_chain(chain_df)
    atm_idx = chain_df[chain_df['strike_price'] == atm].index[0]
    display(chain_df.iloc[atm_idx-5:atm_idx+6][['strike_price', 'ce_ltp', 'ce_delta', 'ce_iv', 'pe_ltp', 'pe_delta', 'pe_iv']])

Total Strikes: 95
Spot Price: ₹25048.65
PCR: nan

Columns: ['strike_price', 'spot_price', 'pcr', 'expiry', 'underlying_key', 'ce_key', 'ce_ltp', 'ce_volume', 'ce_oi', 'ce_prev_oi', 'ce_close', 'ce_bid', 'ce_ask', 'ce_bid_qty', 'ce_ask_qty', 'ce_delta', 'ce_theta', 'ce_gamma', 'ce_vega', 'ce_iv', 'ce_pop', 'pe_key', 'pe_ltp', 'pe_volume', 'pe_oi', 'pe_prev_oi', 'pe_close', 'pe_bid', 'pe_ask', 'pe_bid_qty', 'pe_ask_qty', 'pe_delta', 'pe_theta', 'pe_gamma', 'pe_vega', 'pe_iv', 'pe_pop']

Sample Data (ATM ± 5 strikes):


,strike_price,ce_ltp,ce_delta,ce_iv,pe_ltp,pe_delta,pe_iv
28,24800.0,311.70,0.8300,11.43,22.25,-0.1549,10.73
29,24850.0,268.90,0.7927,11.00,29.60,-0.1971,10.53
30,24900.0,229.80,0.7385,11.05,39.30,-0.2493,10.41
31,24950.0,193.10,0.6844,10.70,52.15,-0.3095,10.31
32,25000.0,157.95,0.6192,10.62,67.95,-0.3762,10.19
33,25050.0,128.25,0.5491,10.64,87.20,-0.4476,9.90
34,25100.0,100.65,0.4772,10.39,110.80,-0.5239,9.97
35,25150.0,78.70,0.4054,10.42,137.35,-0.6015,9.73
36,25200.0,59.25,0.3355,10.35,168.30,-0.6751,9.69
37,25250.0,45.80,0.2741,10.47,204.70,-0.7387,9.84


## 5. Option Greeks

In [13]:
# Get Greeks from Option Chain DataFrame
atm_strike = get_atm_strike_from_chain(chain_df)
print(f"ATM Strike: {atm_strike}")

# CE Greeks
ce_greeks = get_greeks(chain_df, atm_strike, "CE")
print(f"\nCE Greeks:")
for key, value in ce_greeks.items():
    print(f"  {key}: {value}")

# PE Greeks
pe_greeks = get_greeks(chain_df, atm_strike, "PE")
print(f"\nPE Greeks:")
for key, value in pe_greeks.items():
    print(f"  {key}: {value}")

ATM Strike: 25050

CE Greeks:
  delta: 0.5491
  theta: -13.8305
  gamma: 0.0014
  vega: 10.396
  iv: 10.64
  pop: 36.41

PE Greeks:
  delta: -0.4476
  theta: -12.855
  gamma: 0.0015
  vega: 10.3849
  iv: 9.9
  pop: 32.24


In [14]:
# Get Greeks via Direct API Call (for specific instrument)
# First, get instrument key for ATM CE
atm_ce_key = chain_df[chain_df['strike_price'] == atm_strike]['ce_key'].iloc[0]
print(f"ATM CE Key: {atm_ce_key}")

# Fetch Greeks
greeks_api = get_option_greek(access_token, atm_ce_key)
if greeks_api and 'data' in greeks_api:
    print(f"\nGreeks from API:")
    for key, data in greeks_api['data'].items():
        print(f"  Delta: {data.get('delta', 0):.4f}")
        print(f"  Gamma: {data.get('gamma', 0):.4f}")
        print(f"  Theta: {data.get('theta', 0):.4f}")
        print(f"  Vega: {data.get('vega', 0):.4f}")
        print(f"  IV: {data.get('iv', 0):.4f}")

ATM CE Key: NSE_FO|58663

Greeks from API:
  Delta: 0.5491
  Gamma: 0.0014
  Theta: -13.8305
  Vega: 10.3960
  IV: 0.1064


## 6. LTP & OHLC Quotes

In [15]:
# Get LTP for single instrument
ltp_data = get_ltp_quote(access_token, nifty_fut_key)
print(f"LTP Response: {ltp_data}")

LTP Response: {'status': 'success', 'data': {'NSE_FO:NIFTY26JANFUT': {'last_price': 25080.0, 'instrument_token': 'NSE_FO|49229', 'ltq': 65, 'volume': 7055490, 'cp': 25349.8}}}


In [16]:
# Get LTP for multiple instruments
atm_pe_key = chain_df[chain_df['strike_price'] == atm_strike]['pe_key'].iloc[0]

multi_ltp = get_multiple_ltp_quotes(access_token, [atm_ce_key, atm_pe_key])
if multi_ltp and 'data' in multi_ltp:
    for key, data in multi_ltp['data'].items():
        print(f"{key}: ₹{data.get('last_price', 0):.2f}")

NSE_FO:NIFTY26JAN25050CE: ₹128.25
NSE_FO:NIFTY26JAN25050PE: ₹87.20


In [17]:
# Get OHLC Quote
ohlc_data = get_ohlc_quote(access_token, nifty_fut_key, interval="1d")
print(f"OHLC Response: {ohlc_data}")

OHLC Response: {'status': 'success', 'data': {'NSE_FO:NIFTY26JANFUT': {'last_price': 25080.0, 'instrument_token': 'NSE_FO|49229', 'prev_ohlc': None, 'live_ohlc': {'open': 25379.9, 'high': 25379.9, 'low': 25045.7, 'close': 25080.0, 'volume': 7055490, 'ts': 1769106600000}}}}


## 7. Instrument Utilities

In [18]:
# Get Option Instrument Key
strike = atm_strike + 100  # OTM CE
option_key = get_option_instrument_key("NIFTY", strike, "CE", nse_data)
print(f"Option Key for {strike} CE: {option_key}")

Option Key for 25150 CE: NSE_FO|58667


In [19]:
# Get ATM IV (Implied Volatility)
atm_iv = get_atm_iv(chain_df)
print(f"ATM IV: {atm_iv:.2f}%")

ATM IV: 10.64%


## 8. WebSocket Streaming (Live Data)

**Note:** WebSocket requires async execution. Use this in your strategy files.

In [22]:
# Example WebSocket Setup (for reference)
from lib.api.streaming import UpstoxStreamer

# Initialize streamer
streamer = UpstoxStreamer(access_token)

# Define callback
def on_tick(data):
    print(f"Tick: {data}")

# Connect (uncomment to test)
# streamer.connect_market_data(
#     instrument_keys=[nifty_fut_key],
#     mode="ltpc",
#     on_message=on_tick
# )

print("WebSocket example ready (commented out for safety)")

🔑 Access Token configured: eyJ0eXAiOiJKV1QiLCJr... (length: 311)
🔑 Auth Settings OAUTH2 value: Bearer eyJ0eXAiOiJKV1QiLCJrZXl...
WebSocket example ready (commented out for safety)


## 9. Quick Testing Playground

Use this cell to quickly test any API function:

In [ ]:
# Your test code here
